# Regularization and Feature Selection
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Project Overview

Ridge and Lasso regularization applied to the house price dataset, building directly on the `linear_regression_from_scratch.ipynb` from Week 4.

### Main goals:

- Observe how L2 (Ridge) and L1 (Lasso) penalties shrink coefficients as alpha increases.
- Show that Lasso drives some coefficients to exactly zero — implicit feature selection.
- Compare OLS, Ridge, and Lasso on train/test performance.

---

## Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})

## Dataset

House price dataset — same as `house_price_prediction.ipynb` from Week 4.

In [2]:
url = 'https://raw.githubusercontent.com/mohitkhyalia1/sos_2026/refs/heads/main/dataset/house_price_data.csv'
df = pd.read_csv(url)
print(df.shape)
df.head()

In [3]:
X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape}  |  Test: {X_test_sc.shape}')

## Baseline — Ordinary Least Squares

In [4]:
ols = LinearRegression()
ols.fit(X_train_sc, y_train)

train_r2 = r2_score(y_train, ols.predict(X_train_sc))
test_r2  = r2_score(y_test,  ols.predict(X_test_sc))
print(f'OLS — Train R²: {train_r2:.4f}  |  Test R²: {test_r2:.4f}')

## Ridge Regression (L2)

Adds `alpha * Σ w²` to the loss. All coefficients shrink toward zero but none reach exactly zero.

In [5]:
alphas = np.logspace(-3, 4, 80)
ridge_coefs = []
for a in alphas:
    r = Ridge(alpha=a).fit(X_train_sc, y_train)
    ridge_coefs.append(r.coef_)
ridge_coefs = np.array(ridge_coefs)

fig, ax = plt.subplots(figsize=(9, 4))
for j in range(ridge_coefs.shape[1]):
    ax.plot(np.log10(alphas), ridge_coefs[:, j], lw=1.2, alpha=0.75)
ax.axhline(0, color='k', lw=0.8, linestyle='--')
ax.set_xlabel('log10(alpha)')
ax.set_ylabel('Coefficient value')
ax.set_title('Ridge — Coefficient Shrinkage Path')
plt.tight_layout()
plt.show()

**Observation:**
All coefficients shrink smoothly toward zero as alpha increases but never reach exactly zero. Ridge distributes the penalty across all features equally — useful when all features carry some signal.

## Lasso Regression (L1)

Adds `alpha * Σ |w|` to the loss. The L1 geometry causes some coefficients to hit exactly zero, performing automatic feature selection.

In [6]:
lasso_coefs = []
for a in alphas:
    try:
        l = Lasso(alpha=a, max_iter=5000).fit(X_train_sc, y_train)
        lasso_coefs.append(l.coef_)
    except Exception:
        lasso_coefs.append(np.zeros(X_train_sc.shape[1]))
lasso_coefs = np.array(lasso_coefs)

fig, ax = plt.subplots(figsize=(9, 4))
for j in range(lasso_coefs.shape[1]):
    ax.plot(np.log10(alphas), lasso_coefs[:, j], lw=1.2, alpha=0.75)
ax.axhline(0, color='k', lw=0.8, linestyle='--')
ax.set_xlabel('log10(alpha)')
ax.set_ylabel('Coefficient value')
ax.set_title('Lasso — Coefficient Shrinkage Path (features zeroing out)')
plt.tight_layout()
plt.show()

**Observation:**
Lasso coefficients reach exactly zero at different alpha values — each zero-crossing is a feature being dropped from the model. Features that survive to high alpha values are the strongest predictors. This is the key practical difference from Ridge.

## Alpha Selection via Cross-Validation

In [7]:
ridge_cv = RidgeCV(alphas=np.logspace(-3, 4, 100), cv=5)
ridge_cv.fit(X_train_sc, y_train)
print(f'RidgeCV best alpha: {ridge_cv.alpha_:.4f}')

lasso_cv = LassoCV(alphas=np.logspace(-3, 2, 100), cv=5, max_iter=5000)
lasso_cv.fit(X_train_sc, y_train)
print(f'LassoCV best alpha: {lasso_cv.alpha_:.4f}')

## OLS vs Ridge vs Lasso — Comparison

In [8]:
models = {
    'OLS':   (ols, X_train_sc, X_test_sc),
    'Ridge': (Ridge(alpha=ridge_cv.alpha_).fit(X_train_sc, y_train), X_train_sc, X_test_sc),
    'Lasso': (Lasso(alpha=lasso_cv.alpha_, max_iter=5000).fit(X_train_sc, y_train), X_train_sc, X_test_sc),
}

rows = []
for name, (m, Xtr, Xte) in models.items():
    rows.append({
        'Model': name,
        'Train R2': r2_score(y_train, m.predict(Xtr)),
        'Test R2':  r2_score(y_test,  m.predict(Xte)),
        'Non-zero coefs': int(np.sum(np.abs(m.coef_) > 1e-4))
    })

print(pd.DataFrame(rows).to_string(index=False))

## Lasso Feature Selection — Surviving Coefficients

In [9]:
lasso_best = Lasso(alpha=lasso_cv.alpha_, max_iter=5000).fit(X_train_sc, y_train)
coef_df = pd.DataFrame({'feature': X.columns, 'coefficient': lasso_best.coef_})
coef_df = coef_df[coef_df['coefficient'].abs() > 1e-4].sort_values('coefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(7, max(4, len(coef_df) * 0.35)))
colors = ['#2CA02C' if c > 0 else '#C00000' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, alpha=0.85)
ax.set_xlabel('Lasso Coefficient (scaled features)')
ax.set_title(f'Lasso — Non-zero Coefficients (alpha={lasso_cv.alpha_:.4f})')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Observation:**
Lasso retains only the features with genuine predictive power — the remaining coefficients are zero. This list of surviving features is a principled starting point for the feature selection step in the final project pipeline, complementing the tree-based importance ranking from `decision_trees_and_ensembles.ipynb`.